In [ ]:
import geopandas as gpd
import pandas as pd
import fiona
import numpy as np
import os
import time
from shapely import wkb
from shapely.geometry import Point

# Enable KML driver for KMZ reading
fiona.drvsupport.supported_drivers['KML'] = 'rw'
fiona.drvsupport.supported_drivers['LIBKML'] = 'rw'

In [ ]:
shp_path="./data/raw/road_routes/geometria/Geometria_tramos.shp"
traffic_path="./data/processed/road_routes_traffic.parquet"
kmz_path="./data/raw/roads/query.kmz"
output_path="./data/processed/integrated_road_network.parquet"
backbone_output_path="./data/processed/backbone_roads.parquet"
sampling_interval_m=200
buffer_radius_m=50
target_traffic_column="total_max"

In [ ]:
os.chdir('..')
os.getcwd()

In [ ]:
def discretize_line_to_points(gdf, id_col, interval=200):
    """
    Converts LineStrings into a series of Points along their actual path.
    Each point stores the distance from the line's start (m_ref).
    """
    points_data = []
    for _, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        
        # Calculate distances at intervals
        length = geom.length
        if length <= 0:
            distances = [0.0]
        else:
            distances = np.arange(0, length, interval)
            # Ensure we include the end of the line
            if len(distances) == 0 or distances[-1] < length:
                distances = np.append(distances, length)
            
        for d in distances:
            pt = geom.interpolate(d)
            # Carry over all backbone attributes
            entry = row.to_dict()
            entry['geometry'] = pt
            entry['m_ref'] = d
            points_data.append(entry)
            
    return gpd.GeoDataFrame(points_data, crs=gdf.crs)

In [ ]:
gdf_segments = gpd.read_file(shp_path)
if gdf_segments.crs is None:
    gdf_segments.set_crs(3042, inplace=True)
gdf_segments.head()

In [ ]:
df_info = pd.read_parquet(traffic_path)
    
# Selection of traffic columns for processing
if target_traffic_column not in df_info.columns:
    print(f"Warning: {target_traffic_column} not found. Defaulting to 'total_max'.")
    target_traffic_column = "total_max"

# Pick up target total and its corresponding not_short column
base_name = target_traffic_column.replace("total_", "")
not_short_col = f"not_short_{base_name}"

traffic_cols = [target_traffic_column]
if not_short_col in df_info.columns:
    traffic_cols.append(not_short_col)

print(f" - Processing traffic columns: {traffic_cols}")

gdf_segments['id_tramo'] = gdf_segments['id_tramo'].astype(str)
df_info['tramo'] = df_info['tramo'].astype(str)
df_info.head()

In [ ]:
gdf_merged = gdf_segments.merge(df_info, left_on='id_tramo', right_on='tramo')
gdf_merged['seg_length'] = gdf_merged.geometry.length
gdf_merged

In [ ]:
gdf_backbone = gpd.read_file(kmz_path, driver='KML')

# Extract attributes from HTML description
gdf_backbone["route_name"] = gdf_backbone["description"].str.extract(
        r"<td>Carretera</td>\s*<td>([^<]+)</td>", expand=False
    )

gdf_backbone["tipo_via"] = gdf_backbone["description"].str.extract(
    r"<td>Tipo_de_via</td>\s*<td>([^<]+)</td>", expand=False
)

gdf_backbone = gdf_backbone.to_crs(gdf_segments.crs)
gdf_backbone['length_m'] = gdf_backbone.geometry.length
gdf_backbone = gdf_backbone.rename(columns={'id': 'backbone_id'})

In [ ]:
gdf_backbone = gdf_backbone[['backbone_id', 'geometry', 'length_m', 'route_name', 'tipo_via']]
gdf_backbone 

In [ ]:
os.makedirs(os.path.dirname(backbone_output_path), exist_ok=True)
gdf_backbone.to_parquet(backbone_output_path)

In [ ]:
print(f"Discretizing backbones into points (Interval={sampling_interval_m}m)...")
gdf_backbone_pts = discretize_line_to_points(gdf_backbone, 'backbone_id', interval=sampling_interval_m)
gdf_backbone_pts['point_idx'] = gdf_backbone_pts.groupby('backbone_id').cumcount()
gdf_backbone_pts['point_id'] = (
    gdf_backbone_pts['backbone_id'].astype(str) + 
    "_" + 
    gdf_backbone_pts['point_idx'].astype(str)
)
gdf_backbone_pts

In [ ]:
gdf_pts_buffered = gdf_backbone_pts.copy()
gdf_pts_buffered['geometry'] = gdf_pts_buffered.geometry.buffer(buffer_radius_m)
gdf_pts_buffered


In [ ]:
joined = gpd.sjoin(
        gdf_pts_buffered[['point_id', 'backbone_id', 'point_idx', 'geometry']], 
        gdf_merged[['id_tramo', 'geometry', 'seg_length'] + traffic_cols], 
        how='inner', 
        predicate='intersects'
    )
joined

In [ ]:
joined['has_neighbor'] = joined.groupby(['id_tramo', 'backbone_id'])['point_idx'].transform(
            lambda x: x.isin(x + 1) | x.isin(x - 1)
        )
joined = joined[joined['has_neighbor']].copy()
joined = joined.drop(columns=['has_neighbor', 'seg_length'])
joined

In [ ]:
traffic_summary = joined.groupby('point_id')[traffic_cols].sum().reset_index()
traffic_summary

In [ ]:
gdf_final = gdf_backbone_pts.merge(traffic_summary, on='point_id', how='left')
gdf_final[traffic_cols] = gdf_final[traffic_cols].fillna(0)
gdf_final

In [ ]:
segment_counts = joined.groupby('point_id').agg(n_segments=('id_tramo','count'), segments=('id_tramo', list))
segment_counts

In [ ]:
gdf_final = gdf_final.merge(segment_counts, on='point_id', how='left').fillna({'segment_count': 0})
gdf_final

In [ ]:
backbone_id = gdf_final.loc[gdf_final['route_name']=='A-3']['backbone_id'].unique()[1]

In [ ]:
gdf_backbone_plot = gdf_backbone.loc[gdf_backbone['backbone_id']==backbone_id]
gdf_backbone_plot = gdf_backbone_plot.to_crs(epsg=4326)
gdf_backbone_plot

In [ ]:
gdf_final_plot = gdf_final.loc[gdf_final['backbone_id']==backbone_id]
gdf_final_plot = gdf_final_plot.to_crs(epsg=4326)
gdf_final_plot

In [ ]:
gdf_final_plot.sort_values(by='n_segments')

In [ ]:
segments_plot = gdf_merged.loc[gdf_merged['id_tramo'].isin(joined.loc[joined['point_id'].isin(gdf_final_plot['point_id'].unique())]['id_tramo'].unique())]
segments_plot = segments_plot.to_crs(epsg=4326)
segments_plot

In [ ]:
import folium

m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles="CartoDB Positron")

for row in gdf_final_plot.itertuples():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.7,
        tooltip=f'segment count:  {getattr(row, "n_segments", "")}, traffic: {getattr(row, "total_max", "")}'
    ).add_to(m)

# Add Processed Segments
folium.GeoJson(
    gdf_backbone_plot,
    name="Hermes",    
    style_function=lambda x: {'color': 'green', 'weight': 3, 'opacity': 0.5},
    tooltip=folium.GeoJsonTooltip(
        fields=["route_name"],   # column name in your GeoDataFrame
        aliases=["Route:"]
        )
).add_to(m)

folium.GeoJson(
    segments_plot,
    name="Processed Network",
    style_function=lambda x: {'color': 'blue', 'weight': 2, 'opacity': 0.5},
    tooltip=folium.GeoJsonTooltip(
        fields=["tramo"],   # column name in your GeoDataFrame
        aliases=["tramo:"]
        )
).add_to(m)

m

In [ ]:
backbone_points = gpd.read_parquet('data/processed/road_backbone_points.parquet')
backbone_points_plot = backbone_points.copy()
backbone_points_plot = backbone_points.to_crs(epsg=4326)
backbone_points_plot

In [ ]:
backbone_output_path="./data/processed/backbone_roads.parquet"
backbone_roads = gpd.read_parquet(backbone_output_path)

backbone_roads_plot = backbone_roads.copy()
backbone_roads_plot = backbone_roads_plot.to_crs(epsg=4326)
backbone_roads_plot


In [ ]:

# Define a color mapping function based on traffic intensity
def get_traffic_color(intensity):
    if intensity < 12500:
        return "green"   # Low
    elif intensity < 47000:
        return "orange"  # Moderate (Yellow/Orange looks better on white maps)
    else:
        return "red"     # Congested

In [ ]:
import folium

m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles="CartoDB Positron")

for row in backbone_points_plot.itertuples():
    # Use the color function for both border and fill
    color = get_traffic_color(row.total_max)
    
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        tooltip=f"Intensity: {int(row.total_max)}" # Optional: add intensity to tooltip
    ).add_to(m)
# Add Processed Segments
folium.GeoJson(
    backbone_roads_plot,
    name="Hermes",    
    style_function=lambda x: {'color': 'blue', 'weight': 3, 'opacity': 0.5},
    tooltip=folium.GeoJsonTooltip(
        fields=["route_name"],
        aliases=["Route:"]
    )
).add_to(m)
m


In [ ]:
import folium

m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles="CartoDB Positron")

for row in backbone_points_plot.itertuples():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.7
    ).add_to(m)

# Add Processed Segments
folium.GeoJson(
    backbone_roads_plot,
    name="Hermes",    
    style_function=lambda x: {'color': 'blue', 'weight': 3, 'opacity': 0.5},
    tooltip=folium.GeoJsonTooltip(
        fields=["route_name"],   # column name in your GeoDataFrame
        aliases=["Route:"]
        )
).add_to(m)

m